# Agrarian Machine Learning Models - Inference Notebook

This notebook demonstrates how to load the saved machine learning models for **Crop Recommendation** and **Yield Prediction**, preprocess the input parameters, and run predictions.

### Prerequisite Dependencies
Ensure you have the required packages installed. Run the cell below to install any missing dependencies (especially `ipywidgets` for the interactive UI dashboard).

In [12]:
# Run this cell to install the required libraries if you haven't already
# %pip install ipywidgets pandas numpy scikit-learn requests soiltexture

In [13]:
import os
import datetime
import requests
import joblib
import numpy as np
import pandas as pd

# Define paths relative to this notebook (located in notebooks/)
BASE_DIR = os.getcwd()
# If cwd is not notebooks, adjust
if not BASE_DIR.endswith("notebooks"):
    MODEL_DIR = os.path.join(BASE_DIR, "saved_models")
else:
    MODEL_DIR = os.path.join(BASE_DIR, "..", "saved_models")

print(f"Loading models from: {MODEL_DIR}")

Loading models from: c:\Users\lenovo\Documents\GitHub\ml_model_agrarian\notebooks\..\saved_models


In [14]:
# Load the models and transformers
try:
    regressor = joblib.load(os.path.join(MODEL_DIR, "yield_prediction_regressor.joblib"))
    ct_reg = joblib.load(os.path.join(MODEL_DIR, "yield_prediction_transformer.joblib"))
    classifier = joblib.load(os.path.join(MODEL_DIR, "crop_recommendation_classifier.joblib"))
    ct_cla = joblib.load(os.path.join(MODEL_DIR, "crop_recommendation_transformer.joblib"))
    le = joblib.load(os.path.join(MODEL_DIR, "crop_recommendation_labelencoder.joblib"))
    print("All models and transformers loaded successfully!")
except Exception as e:
    print(f"Error loading models. Please verify they exist in {MODEL_DIR}.")
    print(f"Details: {e}")

All models and transformers loaded successfully!


c:\Users\lenovo\Documents\GitHub\ml_model_agrarian\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.7.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\lenovo\Documents\GitHub\ml_model_agrarian\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.7.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\lenovo\Documents\GitHub\ml_model_agrarian\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying

## API Helper Functions

These helper functions fetch environmental data from the NASA POWER API (for 11-year average weather metrics) and the Open-Elevation API (for land elevation).

In [15]:
def generate_fallback_weather_data(start, end):
    """Generates a dummy/simulated pandas DataFrame for fallback/testing purposes."""
    start_dt = datetime.datetime.strptime(start, "%Y%m%d")
    end_dt = datetime.datetime.strptime(end, "%Y%m%d")
    delta = end_dt - start_dt
    
    dates = [start_dt + datetime.timedelta(days=i) for i in range(delta.days + 1)]
    records = []
    for dt in dates:
        # Generate stable, reasonable values for variables
        records.append({
            "YEAR": dt.year,
            "MO": dt.month,
            "DY": dt.day,
            "DOY": dt.timetuple().tm_yday,
            "T2M_MAX": 30.0 + np.sin(dt.timetuple().tm_yday / 50.0) * 3,
            "T2M_MIN": 20.0 + np.sin(dt.timetuple().tm_yday / 50.0) * 2,
            "PRECTOTCORR": float(np.random.choice([0.0, 0.0, 0.0, 1.2, 5.5, 12.0])),
            "RH2M": 80.0 + np.sin(dt.timetuple().tm_yday / 100.0) * 10,
            "ALLSKY_SFC_PAR_TOT": 8.0 + np.cos(dt.timetuple().tm_yday / 100.0) * 2,
            "GWETROOT": 0.5 + np.sin(dt.timetuple().tm_yday / 100.0) * 0.2
        })
    return pd.DataFrame(records)

def fetch_nasa_data(lat, lon, start, end, params):
    """Fetches daily meteorology data from NASA POWER API."""
    url = (
        f"https://power.larc.nasa.gov/api/temporal/daily/point?"
        f"start={start}&end={end}&latitude={lat}&longitude={lon}"
        f"&community=ag&parameters={'%2C'.join(params)}"
        f"&format=csv&header=false"
    )
    try:
        return pd.read_csv(url)
    except Exception as e:
        print(f"Error fetching data from NASA POWER API: {e}")
        print("Using simulated fallback weather data...")
        return generate_fallback_weather_data(start, end)

def get_elevation(lat, lon):
    """Fetches elevation in meters from Open-Elevation API."""
    try:
        r = requests.get(
            f"https://api.open-elevation.com/api/v1/lookup?locations={lat},{lon}",
            timeout=10
        )
        return r.json()["results"][0]["elevation"]
    except Exception as e:
        print(f"Error fetching from Open-Elevation API: {e}")
        print("Using fallback elevation of 450.0m...")
        return 450.0

## Preprocessing Functions

These match the logic in the FastAPI server to prepare features for model prediction.

In [16]:
def yield_data_process(lat, lon, start_date_str, plant_name, base_temp, growing_days):
    year = int(start_date_str[0:4])
    date_obj = datetime.datetime.strptime(start_date_str, "%Y-%m-%d").date()
    doy = date_obj.timetuple().tm_yday

    start_str = (date_obj.replace(year=year - 11)).strftime("%Y%m%d")
    end_str = (date_obj.replace(year=year - 1)).strftime("%Y%m%d")

    params = ["T2M_MAX", "T2M_MIN", "PRECTOTCORR", "GWETROOT", "ALLSKY_SFC_PAR_TOT"]
    df = fetch_nasa_data(lat, lon, start_str, end_str, params)

    end_day = doy + growing_days

    yearly_averages, yearly_avg_temps, yearly_diurnal_ranges, yearly_total_gdds = (
        {p: [] for p in params}, [], [], []
    )

    for i in range(11):
        curr_year = year - 11 + i
        if end_day > 365:
            leap = (curr_year % 4 == 0 and curr_year % 100 != 0) or (curr_year % 400 == 0)
            days_in_year = 366 if leap else 365
            end_day_next = end_day - days_in_year
            
            # Filter and concat data
            df1 = df[(df["YEAR"] == curr_year) & (df["DOY"] >= doy)]
            df2 = df[(df["YEAR"] == curr_year + 1) & (df["DOY"] <= end_day_next)]
            env_df = pd.concat([df1, df2])
        else:
            env_df = df[(df["YEAR"] == curr_year) & (df["DOY"] >= doy) & (df["DOY"] <= end_day)]

        if not env_df.empty:
            for p in params:
                if p in env_df.columns:
                    yearly_averages[p].append(env_df[p].mean())

            avg_temp = (env_df["T2M_MAX"] + env_df["T2M_MIN"]) / 2
            yearly_avg_temps.append(avg_temp.mean())
            yearly_diurnal_ranges.append((env_df["T2M_MAX"] - env_df["T2M_MIN"]).mean())

            gdd = avg_temp - base_temp
            gdd = np.where(gdd < 0, 0, gdd)
            yearly_total_gdds.append(gdd.sum())

    averages = {p: sum(v) / len(v) for p, v in yearly_averages.items() if v}
    averages["avg_temperature"] = sum(yearly_avg_temps) / len(yearly_avg_temps) if yearly_avg_temps else 25.0
    averages["diurnal_temp_range"] = sum(yearly_diurnal_ranges) / len(yearly_diurnal_ranges) if yearly_diurnal_ranges else 8.0
    averages["avg_total_gdd"] = sum(yearly_total_gdds) / len(yearly_total_gdds) if yearly_total_gdds else 1000.0

    return averages

def recommend_data_process(lat, lon):
    year, month, day = datetime.datetime.now().year, datetime.datetime.now().month, datetime.datetime.now().day
    start_str = f"{year-11}0101"
    end_str = f"{year}{month:02d}{day:02d}"

    params = ["T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "ALLSKY_SFC_PAR_TOT"]
    df = fetch_nasa_data(lat, lon, start_str, end_str, params)

    yearly_averages, yearly_avg_temps = {p: [] for p in params}, []
    for i in range(11):
        curr_year = year - 11 + i
        env_df = df[df["YEAR"] == curr_year]
        for p in params:
            if p in env_df.columns:
                yearly_averages[p].append(env_df[p].mean())
        avg_temp = (env_df["T2M_MAX"] + env_df["T2M_MIN"]) / 2
        yearly_avg_temps.append(avg_temp.mean())

    averages = {p: sum(v) / len(v) for p, v in yearly_averages.items() if v}
    averages["avg_temperature"] = sum(yearly_avg_temps) / len(yearly_avg_temps) if yearly_avg_temps else 25.0
    averages["elevation"] = get_elevation(lat, lon)
    return averages

## Direct Inference (Python Code Example)

Here we run crop recommendation and yield prediction using sample parameters programmatically.

In [17]:
# 1. Crop Recommendation Example
print("--- RUNNING CROP RECOMMENDATION ---")
sample_lat = -7.2504
sample_lon = 112.7508
sample_sun_exposure = "Full Sun"  # options: 'Full Sun', 'Partial Shade', 'Full Shade'

# Get and process data
data_cla = recommend_data_process(sample_lat, sample_lon)
inputs_cla = [[
    float(data_cla["avg_temperature"]),
    float(data_cla["PRECTOTCORR"]),
    float(data_cla["RH2M"]),
    float(data_cla["ALLSKY_SFC_PAR_TOT"]),
    float(data_cla["elevation"]),
    sample_sun_exposure
]]

# Transform and predict
transformed_cla = ct_cla.transform(inputs_cla)
pred_cla = classifier.predict(transformed_cla)
recommended_crop = le.inverse_transform(pred_cla)[0]

print(f"Coordinates: {sample_lat}, {sample_lon}")
print(f"Sun Exposure: {sample_sun_exposure}")
print(f"Processed features: {data_cla}")
print(f"Recommended Crop: {recommended_crop}")
print()

# 2. Yield Prediction Example
print("--- RUNNING YIELD PREDICTION ---")
plant_name = "Potatoes"
base_temp = 10.0
growing_days = 120
start_date = "2026-06-01"
area_sq_meters = 1000

data_yield = yield_data_process(sample_lat, sample_lon, start_date, plant_name, base_temp, growing_days)
inputs_yield = [[
    plant_name,
    float(data_yield["T2M_MIN"]),
    float(data_yield["T2M_MAX"]),
    float(data_yield["avg_temperature"]),
    float(data_yield["diurnal_temp_range"]),
    float(data_yield["avg_total_gdd"]),
    float(data_yield["PRECTOTCORR"]),
    float(data_yield["ALLSKY_SFC_PAR_TOT"])
]]

transformed_yield = ct_reg.transform(inputs_yield)
pred_yield_per_unit = regressor.predict(transformed_yield)[0]
total_yield = pred_yield_per_unit * 1000 * area_sq_meters

print(f"Plant: {plant_name}, Start Date: {start_date}, Area: {area_sq_meters} sq.m.")
print(f"Processed features: {data_yield}")
print(f"Predicted Yield: {total_yield:.2f} kg")

--- RUNNING CROP RECOMMENDATION ---


Coordinates: -7.2504, 112.7508
Sun Exposure: Full Sun
Processed features: {'T2M_MAX': np.float64(30.975000823414927), 'T2M_MIN': np.float64(23.2253077394198), 'PRECTOTCORR': np.float64(6.031122933806967), 'RH2M': np.float64(78.0466130970609), 'ALLSKY_SFC_PAR_TOT': np.float64(8.457759889485468), 'avg_temperature': np.float64(27.100154281417364), 'elevation': 6.0}
Recommended Crop: Pepper

--- RUNNING YIELD PREDICTION ---
Plant: Potatoes, Start Date: 2026-06-01, Area: 1000 sq.m.
Processed features: {'T2M_MAX': np.float64(32.119947407963934), 'T2M_MIN': np.float64(22.21444778362134), 'PRECTOTCORR': np.float64(1.2042223891810666), 'GWETROOT': np.float64(0.5476033057851241), 'ALLSKY_SFC_PAR_TOT': np.float64(8.792306536438767), 'avg_temperature': np.float64(27.16719759579264), 'diurnal_temp_range': np.float64(9.9054996243426), 'avg_total_gdd': np.float64(1900.5581818181818)}
Predicted Yield: 779199.00 kg
